# Analizador Localización de Login v4 - Notebook de validación TFM

Este notebook ejecuta la validación experimental del **Analizador Localización de Login v4**.

Flujo:
1. Clonar el repositorio GitHub.
2. Instalar dependencias.
3. Cargar el dataset sintético v4.
4. Ejecutar el analizador batch.
5. Generar métricas, tablas, figuras, Excel y ZIP.

In [ ]:
# 1. Clonar repositorio e instalar dependencias

import os
import shutil

REPO_URL = "https://github.com/dibujoudea-boop/analizador_localizacion_login.git"
REPO_DIR = "analizador_localizacion_login"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL}
%cd {REPO_DIR}

!pip install -q -r requirements.txt

In [ ]:
# 2. Importar librerías y módulos del proyecto

import os
import zipfile
import pandas as pd
import matplotlib.pyplot as plt

from src.batch_analysis import load_dataset, run_batch_analysis, build_summary_tables

In [ ]:
# 3. Cargar dataset sintético v4

DATASET_PATH = "data/logins_sinteticos_v4_analizador_localizacion_login.csv"

df = load_dataset(DATASET_PATH)

print("Dimensiones del dataset:", df.shape)
display(df.head(10))

In [ ]:
# 4. Ejecutar el analizador batch

df_results = run_batch_analysis(df)

print("Eventos procesados:", len(df_results))
display(df_results.head(10))

In [ ]:
# 5. Crear tablas resumen

summary_risk, summary_action, summary_comparison, summary_by_scenario = build_summary_tables(df_results)

display(summary_risk)
display(summary_action)
display(summary_comparison)
display(summary_by_scenario)

In [ ]:
# 6. Métricas principales

total_eventos = len(df_results)
aciertos = (df_results["comparison_result"] == "acierto").sum()
subestimaciones = (df_results["comparison_result"] == "subestimacion").sum()
sobrestimaciones = (df_results["comparison_result"] == "sobrestimacion").sum()
accuracy_global = aciertos / total_eventos

tabla_metricas = pd.DataFrame({
    "Métrica": [
        "Número total de eventos",
        "Eventos clasificados en riesgo bajo",
        "Eventos clasificados en riesgo medio",
        "Eventos clasificados en riesgo alto",
        "Acciones allow",
        "Acciones step_up_mfa",
        "Acciones block_or_review",
        "Aciertos",
        "Subestimaciones",
        "Sobrestimaciones",
        "Accuracy global"
    ],
    "Valor": [
        total_eventos,
        int((df_results["risk_level"] == "bajo").sum()),
        int((df_results["risk_level"] == "medio").sum()),
        int((df_results["risk_level"] == "alto").sum()),
        int((df_results["recommended_action"] == "allow").sum()),
        int((df_results["recommended_action"] == "step_up_mfa").sum()),
        int((df_results["recommended_action"] == "block_or_review").sum()),
        int(aciertos),
        int(subestimaciones),
        int(sobrestimaciones),
        f"{accuracy_global:.2%}".replace(".", ",")
    ]
})

display(tabla_metricas)

In [ ]:
# 7. Matriz de confusión

matriz_confusion = pd.crosstab(
    df_results["expected_result"],
    df_results["risk_level"],
    rownames=["Resultado esperado"],
    colnames=["Resultado obtenido"]
).reindex(
    index=["bajo", "medio", "alto"],
    columns=["bajo", "medio", "alto"],
    fill_value=0
)

display(matriz_confusion)

plt.figure(figsize=(6, 5))
plt.imshow(matriz_confusion, aspect="auto")
plt.title("Matriz de confusión del analizador v4")
plt.xlabel("Resultado obtenido")
plt.ylabel("Resultado esperado")
plt.xticks(range(len(matriz_confusion.columns)), matriz_confusion.columns)
plt.yticks(range(len(matriz_confusion.index)), matriz_confusion.index)

for i in range(len(matriz_confusion.index)):
    for j in range(len(matriz_confusion.columns)):
        plt.text(j, i, matriz_confusion.iloc[i, j], ha="center", va="center")

plt.colorbar(label="Número de eventos")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Mapa de calor: tipo de escenario vs nivel de riesgo

tabla_heatmap = pd.crosstab(df_results["scenario_type"], df_results["risk_level"])
tabla_heatmap = tabla_heatmap.reindex(
    index=["habitual", "legit_unusual", "ambiguous", "suspicious"],
    columns=["bajo", "medio", "alto"],
    fill_value=0
)

display(tabla_heatmap)

plt.figure(figsize=(7, 5))
plt.imshow(tabla_heatmap, aspect="auto")
plt.title("Mapa de calor: tipo de escenario vs nivel de riesgo")
plt.xlabel("Nivel de riesgo")
plt.ylabel("Tipo de escenario")
plt.xticks(range(len(tabla_heatmap.columns)), tabla_heatmap.columns)
plt.yticks(
    range(len(tabla_heatmap.index)),
    ["Habitual", "Legítimo pero inusual", "Ambiguo", "Sospechoso"]
)

for i in range(len(tabla_heatmap.index)):
    for j in range(len(tabla_heatmap.columns)):
        plt.text(j, i, tabla_heatmap.iloc[i, j], ha="center", va="center")

plt.colorbar(label="Número de eventos")
plt.tight_layout()
plt.show()

In [ ]:
# 9. Histograma del score de riesgo

plt.figure(figsize=(7, 4))
df_results["risk_score"].plot(kind="hist", bins=20)
plt.title("Distribución del score de riesgo del analizador v4")
plt.xlabel("Score de riesgo")
plt.ylabel("Cantidad de eventos")
plt.tight_layout()
plt.show()

In [ ]:
# 10. Análisis de umbrales de scoring

def clasificar_por_umbrales(score, umbral_medio, umbral_alto):
    if score >= umbral_alto:
        return "alto"
    elif score >= umbral_medio:
        return "medio"
    else:
        return "bajo"

resultados_umbrales = []
orden = {"bajo": 1, "medio": 2, "alto": 3}

for umbral_medio in [20, 25, 30, 35]:
    for umbral_alto in [45, 50, 55, 60]:
        if umbral_medio >= umbral_alto:
            continue

        clasificacion_temp = df_results["risk_score"].apply(
            lambda x: clasificar_por_umbrales(x, umbral_medio, umbral_alto)
        )

        comparaciones_temp = []

        for esperado, obtenido in zip(df_results["expected_result"], clasificacion_temp):
            esperado = str(esperado).lower()
            obtenido = str(obtenido).lower()

            if esperado == obtenido:
                comparaciones_temp.append("acierto")
            elif orden[obtenido] < orden[esperado]:
                comparaciones_temp.append("subestimacion")
            else:
                comparaciones_temp.append("sobrestimacion")

        comparaciones_temp = pd.Series(comparaciones_temp)

        resultados_umbrales.append({
            "Umbral medio": umbral_medio,
            "Umbral alto": umbral_alto,
            "Aciertos": int((comparaciones_temp == "acierto").sum()),
            "Subestimaciones": int((comparaciones_temp == "subestimacion").sum()),
            "Sobrestimaciones": int((comparaciones_temp == "sobrestimacion").sum()),
            "Accuracy": float((comparaciones_temp == "acierto").mean())
        })

tabla_umbrales = pd.DataFrame(resultados_umbrales)
tabla_umbrales = tabla_umbrales.sort_values(by="Accuracy", ascending=False)

display(tabla_umbrales.head(10))

top_umbrales = tabla_umbrales.head(8).copy()
top_umbrales["Configuración"] = (
    "M" + top_umbrales["Umbral medio"].astype(str) +
    "-A" + top_umbrales["Umbral alto"].astype(str)
)

plt.figure(figsize=(9, 5))
plt.plot(top_umbrales["Configuración"], top_umbrales["Accuracy"], marker="o", label="Accuracy")
plt.title("Comparación de accuracy según umbrales de scoring")
plt.xlabel("Configuración de umbrales")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 11. Exportación automática de tablas, figuras, Excel y ZIP

os.makedirs("outputs", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# Guardar resultados en CSV
df_results.to_csv("outputs/resultados_detallados_v4.csv", index=False)
tabla_metricas.to_csv("outputs/tabla_metricas_v4.csv", index=False)
summary_risk.to_csv("outputs/tabla_riesgo_v4.csv", index=False)
summary_action.to_csv("outputs/tabla_acciones_v4.csv", index=False)
summary_comparison.to_csv("outputs/tabla_comparacion_v4.csv", index=False)
summary_by_scenario.to_csv("outputs/tabla_escenario_vs_riesgo_v4.csv", index=False)
matriz_confusion.to_csv("outputs/matriz_confusion_v4.csv")
tabla_heatmap.to_csv("outputs/tabla_heatmap_escenario_riesgo_v4.csv")
tabla_umbrales.to_csv("outputs/tabla_analisis_umbrales_v4.csv", index=False)

# Guardar tablas en Excel
excel_path = "outputs/resultados_analizador_localizacion_login_v4.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    tabla_metricas.to_excel(writer, sheet_name="metricas", index=False)
    summary_risk.to_excel(writer, sheet_name="riesgo", index=False)
    summary_action.to_excel(writer, sheet_name="acciones", index=False)
    summary_comparison.to_excel(writer, sheet_name="comparacion", index=False)
    summary_by_scenario.to_excel(writer, sheet_name="escenario_riesgo", index=False)
    matriz_confusion.to_excel(writer, sheet_name="matriz_confusion")
    tabla_heatmap.to_excel(writer, sheet_name="heatmap")
    tabla_umbrales.to_excel(writer, sheet_name="umbrales", index=False)
    df_results.to_excel(writer, sheet_name="detalle_eventos", index=False)

print("Archivo Excel generado:", excel_path)

# Figura 5: distribución de riesgo
conteo_riesgo = df_results["risk_level"].value_counts().reindex(["bajo", "medio", "alto"]).fillna(0)

plt.figure(figsize=(7, 4))
conteo_riesgo.plot(kind="bar")
plt.title("Distribución de niveles de riesgo del analizador v4")
plt.xlabel("Nivel de riesgo")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/ilustracion_5_distribucion_riesgo_v4.png", dpi=300)
plt.show()

# Figura 6: riesgo por escenario
tabla_plot = pd.crosstab(df_results["scenario_type"], df_results["risk_level"])
tabla_plot = tabla_plot.reindex(
    index=["ambiguous", "habitual", "legit_unusual", "suspicious"],
    columns=["bajo", "medio", "alto"],
    fill_value=0
)

tabla_plot.plot(kind="bar", figsize=(8, 5))
plt.title("Clasificación de riesgo por tipo de escenario")
plt.xlabel("Tipo de escenario")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/ilustracion_6_riesgo_por_escenario_v4.png", dpi=300)
plt.show()

# Figura 7: comparación esperado vs obtenido
comparacion_plot = (
    df_results["comparison_result"]
    .value_counts()
    .reindex(["acierto", "subestimacion", "sobrestimacion"])
    .fillna(0)
)

plt.figure(figsize=(7, 4))
comparacion_plot.plot(kind="bar")
plt.title("Comparación entre resultado esperado y obtenido")
plt.xlabel("Resultado")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/ilustracion_7_comparacion_esperado_obtenido_v4.png", dpi=300)
plt.show()

# Figura 8: matriz de confusión
plt.figure(figsize=(6, 5))
plt.imshow(matriz_confusion, aspect="auto")
plt.title("Matriz de confusión del analizador v4")
plt.xlabel("Resultado obtenido")
plt.ylabel("Resultado esperado")
plt.xticks(range(len(matriz_confusion.columns)), matriz_confusion.columns)
plt.yticks(range(len(matriz_confusion.index)), matriz_confusion.index)

for i in range(len(matriz_confusion.index)):
    for j in range(len(matriz_confusion.columns)):
        plt.text(j, i, matriz_confusion.iloc[i, j], ha="center", va="center")

plt.colorbar(label="Número de eventos")
plt.tight_layout()
plt.savefig("figures/ilustracion_8_matriz_confusion_v4.png", dpi=300)
plt.show()

# Figura 9: mapa de calor
plt.figure(figsize=(7, 5))
plt.imshow(tabla_heatmap, aspect="auto")
plt.title("Mapa de calor: tipo de escenario vs nivel de riesgo")
plt.xlabel("Nivel de riesgo")
plt.ylabel("Tipo de escenario")
plt.xticks(range(len(tabla_heatmap.columns)), tabla_heatmap.columns)
plt.yticks(
    range(len(tabla_heatmap.index)),
    ["Habitual", "Legítimo pero inusual", "Ambiguo", "Sospechoso"]
)

for i in range(len(tabla_heatmap.index)):
    for j in range(len(tabla_heatmap.columns)):
        plt.text(j, i, tabla_heatmap.iloc[i, j], ha="center", va="center")

plt.colorbar(label="Número de eventos")
plt.tight_layout()
plt.savefig("figures/ilustracion_9_heatmap_escenario_riesgo_v4.png", dpi=300)
plt.show()

# Figura 10: histograma del score
plt.figure(figsize=(7, 4))
df_results["risk_score"].plot(kind="hist", bins=20)
plt.title("Distribución del score de riesgo del analizador v4")
plt.xlabel("Score de riesgo")
plt.ylabel("Cantidad de eventos")
plt.tight_layout()
plt.savefig("figures/ilustracion_10_histograma_score_riesgo_v4.png", dpi=300)
plt.show()

# Figura 11: comparación de umbrales
plt.figure(figsize=(9, 5))
plt.plot(top_umbrales["Configuración"], top_umbrales["Accuracy"], marker="o", label="Accuracy")
plt.title("Comparación de accuracy según umbrales de scoring")
plt.xlabel("Configuración de umbrales")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("figures/ilustracion_11_comparacion_umbrales_scoring_v4.png", dpi=300)
plt.show()

# Comprimir resultados
zip_path = "resultados_tfm_analizador_localizacion_login_v4.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in ["outputs", "figures"]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, arcname=file_path)

print("Archivo comprimido generado:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Descarga automática disponible solo en Google Colab.")